In [ ]:
# Cell 1 — Install
!pip install groq pandas -q
print('Install selesai')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.3 MB/s eta 0:00:00
Install selesai


In [ ]:
# Cell 2 — Imports & Konfigurasi
import re, time, random
import pandas as pd
from groq import Groq

INPUT_CSV    = '/content/gopay_relabeled.csv'   # hasil re-labeling
OUTPUT_CSV   = 'gopay_augmented.csv'            # output akhir
GROQ_API_KEY = 'your_groq_api_key_here'
MODEL        = 'llama-3.1-8b-instant'
N_PARAPHRASE = 4      # jumlah variasi per tweet
TARGET_MIN   = 150    # target minimum sampel per kelas minoritas
DELAY        = 0.6    # detik antar request

client = Groq(api_key=GROQ_API_KEY)
print('Konfigurasi siap')

Konfigurasi siap


In [ ]:
# Cell 3 — Text Cleansing
def clean_tweet(text: str) -> str:
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\bRT\b', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.lower()
    return text

print('Fungsi clean_tweet siap')

Fungsi clean_tweet siap


In [ ]:
# Cell 4 — Load dataset & lihat distribusi
df = pd.read_csv(INPUT_CSV)
df = df[df['label'].isin(['negatif', 'netral', 'positif'])].copy()
df['clean_text'] = df['full_text'].apply(clean_tweet)
df = df[df['clean_text'].str.strip().str.len() > 3].copy()

print(f'Total samples : {len(df)}')
print('Distribusi label:')
print(df['label'].value_counts().to_string())

df_neg = df[df['label'] == 'negatif'].copy()
df_pos = df[df['label'] == 'positif'].copy()
df_net = df[df['label'] == 'netral'].copy()

print(f'\nNegatif : {len(df_neg)} sampel  → target {TARGET_MIN} (+{max(0, TARGET_MIN - len(df_neg))} augmentasi)')
print(f'Positif : {len(df_pos)} sampel  → target {TARGET_MIN} (+{max(0, TARGET_MIN - len(df_pos))} augmentasi)')
print(f'Netral  : {len(df_net)} sampel  → tidak diaugmentasi')

Total samples : 481
Distribusi label:
label
netral     391
negatif     51
positif     39

Negatif : 51 sampel  → target 150 (+99 augmentasi)
Positif : 39 sampel  → target 150 (+111 augmentasi)
Netral  : 391 sampel  → tidak diaugmentasi


In [ ]:
# Cell 5 — Fungsi parafrase via Groq
SYSTEM_PROMPT = (
    'Kamu adalah sistem augmentasi data NLP untuk Bahasa Indonesia.\n'
    'Tugasmu: buat TEPAT {n} variasi parafrase dari tweet berikut.\n'
    'Aturan WAJIB:\n'
    '- Pertahankan makna dan sentimen PERSIS SAMA dengan aslinya\n'
    '- Gunakan Bahasa Indonesia informal/gaul seperti aslinya\n'
    '- Boleh ganti kata/struktur kalimat, tapi jangan ubah inti pesan\n'
    '- Output HANYA {n} baris teks, satu variasi per baris\n'
    '- Jangan tambahkan nomor, tanda baca, atau penjelasan apapun\n'
    '- Jangan menyebut GoPay jika tidak ada di teks asli'
)


def paraphrase_tweet(text: str, n: int = N_PARAPHRASE, retries: int = 3) -> list:
    prompt = SYSTEM_PROMPT.replace('{n}', str(n))
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': prompt},
                    {'role': 'user',   'content': f'Tweet asli: {text}'}
                ],
                max_tokens=400,
                temperature=0.8,   # sedikit kreatif untuk variasi
            )
            raw = response.choices[0].message.content.strip()
            lines = [l.strip() for l in raw.split('\n') if l.strip()]
            # Ambil tepat n baris
            lines = lines[:n]
            # Filter baris yang terlalu mirip atau terlalu pendek
            lines = [l for l in lines if len(l) > 10 and l.lower() != text.lower()]
            return lines if lines else []
        except Exception as e:
            print(f'  Error attempt {attempt+1}: {e}')
            time.sleep(2 ** attempt)
    return []


# Test cepat
sample = df_neg['clean_text'].iloc[0]
print(f'Teks asli   : {sample}')
print('Parafrase   :')
test_results = paraphrase_tweet(sample, n=2)
for r in test_results:
    print(f'  -> {r}')
print('\nFungsi paraphrase_tweet siap')

Teks asli   : kenapa sih alfamart kebanyakan malas kali transaksi uang digital seperti dana gopay shopeepay dll alasan nya sama semua lagi gak konek lah lagi pertukaran shift lah lagi gangguan jaringan lah
Parafrase   :
  -> Variasi 1: kenapa sih alfamart malas melayani transaksi uang digital? gak konek, pertukaran shift, dan gangguan jaringan selalu jadi alasan!
  -> Variasi 2: kok alfamart sering malas nggak sih? kayak gak bisa nggak konek, pertukaran shift, dan gangguan jaringan selalu bikin transaksi uang digital nggak lancar!

Fungsi paraphrase_tweet siap


In [ ]:
# Cell 6 — Augmentasi negatif & positif
def augment_class(df_class, label, target):
    current   = len(df_class)
    needed    = max(0, target - current)
    if needed == 0:
        print(f'{label}: sudah {current} sampel, tidak perlu augmentasi')
        return []

    # Hitung berapa kali tiap tweet perlu diparafrase
    # Distribusikan kebutuhan secara merata ke semua sampel
    texts    = df_class['clean_text'].tolist()
    aug_rows = []
    idx      = 0

    print(f'\n{label.upper()} — {current} asli, butuh {needed} augmentasi')
    print('-' * 50)

    while len(aug_rows) < needed:
        text   = texts[idx % len(texts)]
        n_req  = min(N_PARAPHRASE, needed - len(aug_rows))
        paraphrases = paraphrase_tweet(text, n=n_req)

        for p in paraphrases:
            aug_rows.append({'full_text': p, 'clean_text': p, 'label': label})
            if len(aug_rows) >= needed:
                break

        done = len(aug_rows)
        print(f'  [{done:3d}/{needed}] dari: {text[:60]}...')

        # Auto-save setiap 20 augmentasi
        if done % 20 == 0 and done > 0:
            print(f'  Auto-saved checkpoint ({done}/{needed})')

        idx += 1
        time.sleep(DELAY)

    return aug_rows


aug_negatif = augment_class(df_neg, 'negatif', TARGET_MIN)
aug_positif = augment_class(df_pos, 'positif', TARGET_MIN)

print(f'\nAugmentasi selesai!')
print(f'  Negatif baru : {len(aug_negatif)}')
print(f'  Positif baru : {len(aug_positif)}')


NEGATIF — 51 asli, butuh 99 augmentasi
--------------------------------------------------
  [  4/99] dari: kenapa sih alfamart kebanyakan malas kali transaksi uang dig...
  [  8/99] dari: apakah aktifasi uang digital harus pakai gak bisa pake gopay...
  [ 12/99] dari: kalo narik uang jadi digital rupiah masih kena admin meski c...
  [ 16/99] dari: baru sadar pergerakan uang digital cukup ngeri ternyata apal...
  [ 20/99] dari: dana tolong saya transaksi topup uang digital gopay lewat da...
  Auto-saved checkpoint (20/99)
  [ 24/99] dari: yes kalo dari penyataan mereka dr kmaren ya kynya sebodo ama...
  [ 28/99] dari: jangan pernah setor simpan uang dalam bentuk digital yes ada...
  [ 32/99] dari: uang digital skrg macemmacem dehhh kdg gue buka krn dpt prom...
  [ 36/99] dari: aku kan niat beli dogllars yang itu kan soalenya emang ngepa...
  [ 40/99] dari: suatu saat tidak ada lagi kantor bank apalagi mendapat senyu...
  Auto-saved checkpoint (40/99)
  [ 44/99] dari: lihat skrg semua d

In [ ]:
# Cell 7 — Gabung & simpan dataset final
df_aug_neg = pd.DataFrame(aug_negatif)
df_aug_pos = pd.DataFrame(aug_positif)

# Gabungkan: data asli + augmentasi
df_final = pd.concat([
    df[['full_text', 'clean_text', 'label']],
    df_aug_neg,
    df_aug_pos
], ignore_index=True)

# Shuffle
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

df_final.to_csv(OUTPUT_CSV, index=False)

print('=' * 50)
print('  DATASET FINAL')
print('=' * 50)
print(f'Total sampel  : {len(df_final)}')
print('\nDistribusi label:')
print(df_final['label'].value_counts().to_string())
print(f'\nDisimpan ke   : {OUTPUT_CSV}')
print('=' * 50)

try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    print('File didownload ke komputer')
except ImportError:
    print('Bukan Colab — download manual dari Files panel')

  DATASET FINAL
Total sampel  : 691

Distribusi label:
label
netral     391
positif    150
negatif    150

Disimpan ke   : gopay_augmented.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File didownload ke komputer
